# Function approximation in RL — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def softmax(z):
    z=np.asarray(z,dtype=float); e=np.exp(z-z.max()); return e/e.sum()
def discount(rewards,gamma=0.9):
    return sum((gamma**t)*r for t,r in enumerate(rewards))
def moving_avg(x,n=5):
    x=np.asarray(x,dtype=float); return np.convolve(x,np.ones(n)/n,mode='valid')
print('setup ok')

## Approximating values with features instead of tables
Function approximation shares evidence across states that have similar features.

In [ ]:
# 1) discounted return: delayed reward still counts, but less than immediate reward
rewards=np.array([1.,0.,2.]); gamma=0.9
terms=np.array([(gamma**t)*r for t,r in enumerate(rewards)]); G=terms.sum()
plt.figure(figsize=(6,3)); plt.bar(range(3),terms); plt.xlabel('time'); plt.ylabel('discounted reward'); plt.title(f'return = {G:.3f}')
assert abs(G-2.62)<1e-9

In [ ]:
# 2) one Bellman-style update moves partway toward a bootstrap target
q_old=0.4; target=1+0.9*0.8; alpha=0.5; q_new=q_old+alpha*(target-q_old)
plt.figure(figsize=(6,3)); plt.bar(['old Q','target','new Q'],[q_old,target,q_new],color=['tab:blue','tab:orange','tab:green']); plt.ylim(0,2)
plt.title('partial step toward reward + discounted future')
assert abs(q_new-1.06)<1e-9

In [ ]:
# 3) exact policy evaluation in a two-state MDP
P=np.array([[0.2,0.8],[0.0,1.0]]); r=np.array([0.5,1.0]); gamma=0.9
V=np.linalg.solve(np.eye(2)-gamma*P,r)
plt.figure(figsize=(6,3)); plt.bar(['state 0','state 1'],V); plt.ylabel('V'); plt.title('solving (I-gamma P)V=r')
assert np.allclose(V,[9.390243902439025,10.0])

In [ ]:
# 4) value iteration converges by repeated local backups
P0=np.array([[0.8,0.2],[0.1,0.9]]); P1=np.array([[0.1,0.9],[0.7,0.3]]); R=np.array([[0.0,1.0],[0.2,0.4]])
V=np.zeros(2); hist=[]
for _ in range(12):
    Q=np.stack([R[:,0]+0.9*P0@V, R[:,1]+0.9*P1@V],axis=1); V=Q.max(axis=1); hist.append(V.copy())
hist=np.array(hist); plt.figure(figsize=(6,3)); plt.plot(hist); plt.xlabel('sweep'); plt.ylabel('V'); plt.title('successive Bellman optimality backups')
assert hist[-1,0]>hist[0,0] and hist[-1,1]>hist[0,1]

In [ ]:
# 5) a tiny gridworld path makes return visible along space
path=np.array([[0,0],[0,1],[1,1],[1,2]]); rewards=np.array([-0.1,-0.1,-0.1,1.0]); gamma=0.9
G=discount(rewards,gamma); grid=np.full((2,3),np.nan)
for t,(i,j) in enumerate(path): grid[i,j]=(gamma**t)*rewards[t]
plt.figure(figsize=(5,3)); plt.imshow(grid,cmap='viridis'); plt.colorbar(label='discounted reward'); plt.title(f'path return = {G:.3f}')
assert abs(G-0.458)<1e-9